<a href="https://colab.research.google.com/github/YOUR-USERNAME/bags-vectors-transformers/blob/main/day4/notebooks/1_llm_annotation_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 4 — LLM Annotation & Distillation with a Local Open Model

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the final notebook, and it ties the whole course together. We use a **small open-weight
LLM** to **annotate** text, then **distill** its knowledge into a fast BERT classifier — the
"best of both worlds" idea from the lecture.

We deliberately use an **open model you run yourself** (not a commercial API), so your data
never leaves your control — the responsible-use principle from Part 2.

By the end you will be able to:

- Load and run a **small open LLM** (Qwen2.5-1.5B) locally via Hugging Face
- **Prompt** it to annotate text (zero-shot and few-shot)
- **Validate** its labels against a human-coded gold standard
- **Distill**: use LLM labels to fine-tune a BERT classifier for cheap, fast, local inference
- Understand how you'd do the same with **Ollama** on your own machine

> **Turn on the GPU!** *Runtime → Change runtime type → T4 GPU.* An LLM on CPU is painfully slow.
> The first model download is ~3 GB, so give it a minute.


## 0. Setup

We use an **ungated, Apache-2.0 licensed** model (`Qwen2.5-1.5B-Instruct`) — no account,
token, or license acceptance needed. It's small enough for free Colab but a capable
instruction-follower.


In [ ]:
!pip install transformers datasets accelerate -q

import torch
import pandas as pd
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device.upper())
if device == "cpu":
    print("⚠️  No GPU — the LLM will be very slow. Enable a GPU runtime!")

## 1. Loading a local open LLM

We load Qwen2.5-1.5B-Instruct with the `text-generation` pipeline. This is a **real LLM**,
running **on your machine** (Colab's GPU) — nothing is sent to any company's servers.


In [ ]:
# Load the open LLM (downloads ~3 GB the first time)
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype="auto",
    device_map="auto",
)
print("LLM loaded and running locally!")

Instruction-tuned models expect a **chat format** with roles (`system`, `user`). Let's
send a simple test message to confirm it works.


In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "In one sentence, what is computational text analysis?"},
]

output = llm(messages, max_new_tokens=80)
print(output[0]["generated_text"][-1]["content"])

## 2. Annotation, take 1: zero-shot

Now the real task. We'll annotate **policy bill titles** by their policy area — the same data
from earlier notebooks. We give the LLM our "codebook" as a **prompt** and ask for a label.

First, the data.


In [ ]:
from datasets import load_dataset

bills = load_dataset("dreamproit/bill_labels_us", split="train").to_pandas()
bills = bills.rename(columns={"title": "text"})[["text", "policy_area"]].dropna()

# Keep four clear, common policy areas for a clean demo
AREAS = ["Health", "Education", "Taxation", "Armed Forces and National Security"]
bills = bills[bills["policy_area"].isin(AREAS)]

# A small sample to annotate (LLMs are slow — this is the whole point of distillation later!)
sample = bills.groupby("policy_area", group_keys=False).apply(
    lambda g: g.sample(min(len(g), 15), random_state=42)
).reset_index(drop=True)

print(f"Annotating {len(sample)} bill titles across {len(AREAS)} policy areas.")
sample.head()

Here's our annotation function. Notice how the **prompt** is really a compact **codebook**:
it defines the task, lists the valid labels, and asks for a clean output.


In [ ]:
def annotate_zero_shot(text):
    """Ask the LLM to classify one bill title into a policy area."""
    system = (
        "You are an expert political science coder. "
        "Classify each bill title into exactly ONE of these policy areas:\n"
        "- Health\n- Education\n- Taxation\n- Armed Forces and National Security\n"
        "Reply with ONLY the policy area name, nothing else."
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"Bill title: {text}"},
    ]
    out = llm(messages, max_new_tokens=15, do_sample=False)  # deterministic
    return out[0]["generated_text"][-1]["content"].strip()

# Test on a few titles
for text in sample["text"].head(3):
    print(f"'{text}'\n  -> {annotate_zero_shot(text)}\n")

> **✏️ Exercise 1**
>
> Run `annotate_zero_shot` on three bill titles of your own invention. Does the model label
> them correctly? Try an **ambiguous** title (e.g. one about military hospitals — Health or
> Armed Forces?) and see what it picks.


In [ ]:
# Your code here


## 3. The crucial step: validate against the gold standard

The lecture's golden rule: an LLM annotator is a **measurement instrument**, and instruments
must be **validated**. We have human labels (`policy_area`) from the Congressional Research
Service — our **gold standard**. Let's measure how well the LLM agrees.


In [ ]:
# Annotate the whole sample (this takes a minute — LLMs are slow!)
sample["llm_label"] = sample["text"].apply(annotate_zero_shot)

# The LLM's output may not exactly match our label strings — clean it up
def normalize(label):
    label = label.strip()
    for area in AREAS:
        if area.lower() in label.lower():
            return area
    return label   # unrecognized

sample["llm_clean"] = sample["llm_label"].apply(normalize)
sample[["text", "policy_area", "llm_clean"]].head(10)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(sample["policy_area"], sample["llm_clean"])
print(f"LLM agreement with human coders: {acc:.3f}\n")
print(classification_report(sample["policy_area"], sample["llm_clean"],
                            labels=AREAS, zero_division=0))

This agreement score is exactly what you'd **report in a paper** to justify using the LLM
as an annotator. If it's high, you have evidence the LLM is a reasonable coder for this task.
If it's low, you'd revise the prompt — or not trust the LLM here.

> **✏️ Exercise 2**
>
> Look at the rows where `policy_area` and `llm_clean` disagree. Are the disagreements
> *reasonable* (genuinely ambiguous titles) or *errors*? This qualitative check matters as much
> as the accuracy number.


In [ ]:
# Your code here — hint: sample[sample['policy_area'] != sample['llm_clean']]


## 4. Improving annotation: few-shot prompting

We can often improve the LLM by showing it a few **worked examples** first (few-shot
prompting, from the lecture). This anchors its understanding of the labels.


In [ ]:
def annotate_few_shot(text):
    """Classify with a few examples in the prompt to guide the model."""
    system = (
        "You are an expert political science coder. Classify each bill title into exactly "
        "ONE of: Health, Education, Taxation, Armed Forces and National Security. "
        "Reply with ONLY the policy area name."
    )
    # A few labeled examples (the 'shots')
    examples = [
        ("A bill to fund cancer research at national institutes.", "Health"),
        ("An act to reduce student loan interest rates.", "Education"),
        ("A bill to adjust corporate tax brackets.", "Taxation"),
        ("An act to modernize the air force fleet.", "Armed Forces and National Security"),
    ]
    messages = [{"role": "system", "content": system}]
    for ex_text, ex_label in examples:
        messages.append({"role": "user", "content": f"Bill title: {ex_text}"})
        messages.append({"role": "assistant", "content": ex_label})
    messages.append({"role": "user", "content": f"Bill title: {text}"})

    out = llm(messages, max_new_tokens=15, do_sample=False)
    return out[0]["generated_text"][-1]["content"].strip()

# Compare zero-shot vs few-shot on the sample
sample["llm_fewshot"] = sample["text"].apply(lambda t: normalize(annotate_few_shot(t)))
acc_few = accuracy_score(sample["policy_area"], sample["llm_fewshot"])
print(f"Zero-shot agreement: {acc:.3f}")
print(f"Few-shot agreement:  {acc_few:.3f}")

> **✏️ Exercise 3**
>
> Did few-shot help? Try changing the examples — use *harder* or *edge-case* examples instead
> of obvious ones. Does that help more? (In practice, choosing good few-shot examples is a real
> part of the craft.)


In [ ]:
# Your code here


## 5. The payoff: distillation

Here's the problem with what we just did: the LLM is **slow**. Annotating 60 titles took a
while; annotating your **whole corpus** of thousands would take hours and lots of compute.

The **distillation** idea from the lecture solves this:

1. The **LLM** (teacher) labels a subset — we just did this.
2. We **fine-tune a small BERT** (student) on those labels.
3. The **BERT** then labels everything else — fast, cheap, local, reproducible.

Let's do it. First, the LLM labels a bigger training subset. *(We keep it modest here for
time; in practice you'd label more.)*


In [ ]:
# LLM-label a training set (the teacher's output becomes our training labels)
train_pool = bills.groupby("policy_area", group_keys=False).apply(
    lambda g: g.sample(min(len(g), 60), random_state=1)
).reset_index(drop=True)

print(f"LLM is labeling {len(train_pool)} titles (the teacher step)... this takes a few minutes.")
train_pool["llm_label"] = train_pool["text"].apply(lambda t: normalize(annotate_few_shot(t)))

# Keep only rows the LLM labeled with a valid area
train_pool = train_pool[train_pool["llm_label"].isin(AREAS)].reset_index(drop=True)
print("Usable LLM-labeled examples:", len(train_pool))

Now we fine-tune DistilBERT on the **LLM's labels** (not the human labels!). The student
learns to imitate the teacher.


In [ ]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.model_selection import train_test_split

# Map labels to integers
label2id = {a: i for i, a in enumerate(AREAS)}
id2label = {i: a for a, i in label2id.items()}
train_pool["label"] = train_pool["llm_label"].map(label2id)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tok(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=32)

tr, te = train_test_split(train_pool, test_size=0.2, random_state=42, stratify=train_pool["label"])
train_ds = Dataset.from_pandas(tr).map(tok, batched=True)
test_ds = Dataset.from_pandas(te).map(tok, batched=True)

student = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(AREAS),
    id2label=id2label, label2id=label2id,
)
print("Student model ready to learn from the teacher's labels.")

In [ ]:
args = TrainingArguments(
    output_dir="./distilled",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="no",
    logging_steps=10,
    report_to="none",
)
trainer = Trainer(model=student, args=args, train_dataset=train_ds)
trainer.train()
print("Distillation complete — the student has learned from the LLM.")

### Did it work? Validate the student against the *human* gold standard

The real test: does our distilled BERT — trained only on *LLM* labels — agree with the
*human* coders? We evaluate it on our original held-out `sample` (which has human labels).


In [ ]:
import numpy as np

student_clf = pipeline("text-classification", model=student, tokenizer=tokenizer, device=0 if device=="cuda" else -1)

def student_predict(text):
    return student_clf(text, truncation=True, max_length=32)[0]["label"]

sample["student_label"] = sample["text"].apply(student_predict)
acc_student = accuracy_score(sample["policy_area"], sample["student_label"])

print("Agreement with HUMAN gold standard:")
print(f"  LLM (teacher):      {acc_few:.3f}")
print(f"  DistilBERT (student): {acc_student:.3f}")
print()
print("The student runs ~100x faster than the LLM — for nearly the same quality.")

**This is the whole idea in one number.** The distilled BERT approaches the LLM's agreement
with human coders — but it runs in **milliseconds**, on a **CPU**, with **no API**, fully
**reproducible**. You paid the LLM cost once, for a subset; the student handles the rest forever.

> **✏️ Exercise 4**
>
> Time how long the LLM takes to label 10 titles versus the distilled BERT. *(Hint: `import
> time`.)* The speed difference is the practical case for distillation.


In [ ]:
# Your code here


## 6. Doing this for real: Ollama on your own machine

In this notebook we ran the LLM via Hugging Face inside Colab. On **your own computer**, the
easiest way to run open models locally is **[Ollama](https://ollama.com)** — a tool that makes
local LLMs as simple as one command.

**You can't easily run Ollama inside Colab** (it needs a background server and local install),
but here's the whole workflow for your own machine. After installing Ollama:

```bash
# In your terminal — download and run a model
ollama pull qwen2.5:1.5b
ollama run qwen2.5:1.5b
```

And from Python, Ollama exposes a simple API:

```python
# pip install ollama
import ollama

response = ollama.chat(model="qwen2.5:1.5b", messages=[
    {"role": "system", "content": "You are an expert political science coder..."},
    {"role": "user", "content": "Bill title: A bill to fund rural hospitals."},
])
print(response["message"]["content"])
```

The **concepts are identical** to what we did here — prompt, annotate, validate, distill. Ollama
just makes the "run it locally" part trivial, handles model downloads and quantization
automatically, and keeps everything on your machine. For sensitive data (interviews, patient
records), this is the responsible choice.


## Wrap-up

You've now completed the final piece, and the whole course:

- Ran a **local open LLM** — no API, no data leaving your control
- Used it to **annotate** policy text (zero-shot and few-shot)
- **Validated** its labels against a human gold standard — the essential discipline
- **Distilled** the LLM into a fast, cheap, reproducible BERT classifier
- Learned how to run this **truly locally with Ollama**

This is arguably the state of the art for practical computational social science: **LLM quality,
BERT efficiency, full transparency and control.**

### The whole journey

From **bags** of words → static **vectors** → contextual **transformers** → **LLMs** and
distillation. You now have the complete modern toolkit — and, just as importantly, the judgment
to **choose the right tool** and **validate what it tells you**.

### Optional challenge

The distilled model was trained on **LLM labels**, which may contain errors. Train a *second*
DistilBERT on the **human** labels for the same titles, and compare the two on the held-out
sample. How much did using LLM labels (instead of human ones) cost you in accuracy? This
quantifies the price of the "teacher's" imperfections.


In [ ]:
# Optional challenge — your code here
